In [ ]:
import pandas as pd


customers = pd.read_csv('olist-raw-csv/olist_customers_dataset.csv')
orders = pd.read_csv('olist-raw-csv/olist_orders_dataset.csv')
order_items = pd.read_csv('olist-raw-csv/olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist-raw-csv/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist-raw-csv/olist_order_reviews_dataset.csv')
products = pd.read_csv('olist-raw-csv/olist_products_dataset.csv')
sellers = pd.read_csv('olist-raw-csv/olist_sellers_dataset.csv')
# geolocation = pd.read_csv('olist-raw-csv/olist_geolocation_dataset.csv')
product_category_translation = pd.read_csv('olist-raw-csv/product_category_name_translation.csv')


In [ ]:
import random
from faker import Faker
import hashlib

fake = Faker()

def fake_city_code(city: str, state: str) -> int:
    state = state.upper().strip()

    key = f"{city}_{state}"

    h = hashlib.md5(key.encode()).hexdigest()
    
    return int(h[:7], 16) % 10_000_000

customers_tbl = pd.DataFrame()
customers_tbl["MaKH"] = customers["customer_id"]
customers_tbl["MaThanhPho"] = customers["customer_zip_code_prefix"]

first_order_per_customer = (
    orders.groupby("customer_id", as_index=False)["order_purchase_timestamp"]
    .min()
    .rename(columns={"order_purchase_timestamp": "NgayDatDauTien"})
)

khachhang_tbl = customers.merge(first_order_per_customer, on="customer_id", how="left")
khachhang_tbl["MaKH"] = khachhang_tbl["customer_id"]
khachhang_tbl["TenKH"] = None
khachhang_tbl["TenKH"] = khachhang_tbl["TenKH"].apply(lambda x: fake.name())
khachhang_tbl["MaThanhPho"] = None
khachhang_tbl["MaThanhPho"] = khachhang_tbl.apply(lambda row: fake_city_code(row["customer_city"], row["customer_state"]), axis=1)
khachhang_tbl = khachhang_tbl[["MaKH", "TenKH", "NgayDatDauTien", "MaThanhPho"]]


most_popular_seller = order_items.merge(sellers, on="seller_id", how="left")
most_popular_seller = most_popular_seller.groupby(["seller_id", "seller_state", "seller_city"], as_index=False).agg({"order_item_id": "count"})
most_popular_seller = most_popular_seller.loc[most_popular_seller.groupby(["seller_state", "seller_city"])["order_item_id"].idxmax()]
most_popular_seller = most_popular_seller.merge(sellers, on=["seller_id", "seller_state", "seller_city"], how="left")

brazil_states = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}


vanphong_tbl = most_popular_seller.copy()
vanphong_tbl["TenThanhPho"] = most_popular_seller["seller_city"]
vanphong_tbl["MaThanhPho"] = None
vanphong_tbl["MaThanhPho"] = vanphong_tbl.apply(lambda row: fake_city_code(row["TenThanhPho"], row["seller_state"]), axis=1)
vanphong_tbl["DiaChiVP"] = most_popular_seller["seller_city"] + ", " + most_popular_seller["seller_state"]
vanphong_tbl["Bang"] = most_popular_seller["seller_state"].map(brazil_states)
vanphong_tbl = vanphong_tbl[["MaThanhPho", "TenThanhPho", "DiaChiVP", "Bang"]]

cuahang_tbl = sellers.copy()
cuahang_tbl["MaCuaHang"] = cuahang_tbl["seller_id"]
cuahang_tbl["MaThanhPho"] = None
cuahang_tbl["MaThanhPho"] = cuahang_tbl.apply(lambda row: fake_city_code(row["seller_city"], row["seller_state"]), axis=1)


def random_brazil_phone_mobile():
    area_code = random.choice(["11", "21", "31", "41", "51"])  # sample
    number = "9" + "".join([str(random.randint(0, 9)) for _ in range(8)])
    
    return f"+55 ({area_code}) {number[:5]}-{number[5:]}"

cuahang_tbl["SoDienThoai"] = None
cuahang_tbl["SoDienThoai"] = cuahang_tbl["SoDienThoai"].apply(lambda x: random_brazil_phone_mobile())
cuahang_tbl = cuahang_tbl[["MaCuaHang", "MaThanhPho", "SoDienThoai"]]



khachdulich_tbl = pd.DataFrame(columns=["MaKH", "HuongDanVien"])
khachbuudien_tbl = pd.DataFrame(columns=["MaKH", "DiaChiBuuDien"])


prd_cat_map = product_category_translation.set_index("product_category_name")["product_category_name_english"].to_dict()

mathang_tbl = products.copy()
mathang_tbl["MaMH"] = mathang_tbl["product_id"]
mathang_tbl["MoTa"] = mathang_tbl["product_category_name"].apply(lambda x: 
    " ".join(prd_cat_map[x].split("_"))
    if x in prd_cat_map else None)

size_cols = ["product_length_cm", "product_height_cm", "product_width_cm"]
if all(col in mathang_tbl.columns for col in size_cols):
    mathang_tbl["KichThuoc"] = (
        mathang_tbl["product_length_cm"].astype(str)
        + "x" + mathang_tbl["product_height_cm"].astype(str)
        + "x" + mathang_tbl["product_width_cm"].astype(str)
    )
else:
    mathang_tbl["KichThuoc"] = None



if "product_weight_g" in mathang_tbl.columns:
    mathang_tbl["TrongLuong"] = mathang_tbl["product_weight_g"] / 1000.0
else:
    mathang_tbl["TrongLuong"] = None
mathang_tbl = mathang_tbl[["MaMH", "MoTa", "KichThuoc", "TrongLuong"]]


mathangluutru_tbl = (
    order_items.groupby(["seller_id", "product_id"], as_index=False)
    .agg(SoLuongTon=("order_item_id", "count"))
)
mathangluutru_tbl["MaCuaHang"] = mathangluutru_tbl["seller_id"]
mathangluutru_tbl["MaMH"] = mathangluutru_tbl["product_id"]
mathangluutru_tbl = mathangluutru_tbl[["MaCuaHang", "MaMH", "SoLuongTon"]]



dondathang_tbl = orders.copy()
dondathang_tbl["MaDon"] = dondathang_tbl["order_id"]
dondathang_tbl["NgayDat"] = dondathang_tbl["order_purchase_timestamp"]
dondathang_tbl["MaKH"] = dondathang_tbl["customer_id"]
dondathang_tbl = dondathang_tbl[["MaDon", "NgayDat", "MaKH"]]



order_items_grouped = (
    order_items.groupby(["order_id", "product_id"], as_index=False)
    .agg(
        SoLuongDat=("order_item_id", "count"),
        Gia=("price", "sum"),
    )
)

mathangduocdat_tbl = order_items_grouped.copy()
mathangduocdat_tbl["MaDon"] = mathangduocdat_tbl["order_id"]
mathangduocdat_tbl["MaMH"] = mathangduocdat_tbl["product_id"]
mathangduocdat_tbl = mathangduocdat_tbl[["MaDon", "MaMH", "SoLuongDat", "Gia"]]


In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import urllib

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
db = os.getenv("DB_NAME")

params = urllib.parse.quote_plus(
    f"DRIVER=ODBC Driver 17 for SQL Server;"
    f"SERVER={host};"
    f"DATABASE={db};"
    f"UID={user};"
    f"PWD={password}"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [53]:
vanphong_tbl.to_sql('VanPhongDaiDien', con=engine, if_exists='append', index=False)
cuahang_tbl.to_sql('CuaHang', con=engine, if_exists='append', index=False)
khachhang_tbl.to_sql('KhachHang', con=engine, if_exists='append', index=False)
khachdulich_tbl.to_sql('KhachDuLich', con=engine, if_exists='append', index=False)
khachbuudien_tbl.to_sql('KhachBuuDien', con=engine, if_exists='append', index=False)
mathang_tbl.to_sql('MatHang', con=engine, if_exists='append', index=False)
mathangluutru_tbl.to_sql('MatHangLuuTru', con=engine, if_exists='append', index=False)
dondathang_tbl.to_sql('DonDatHang', con=engine, if_exists='append', index=False)
mathangduocdat_tbl.to_sql('MatHangDuocDat', con=engine, if_exists='append', index=False)

245